# FastRP Performance Benchmark

This notebook benchmarks all three underlying implementations of the `fastrp` Python bridge:

| Path | Input | Scale |
|------|-------|-------|
| **CSR** | `scipy.sparse.csr_matrix` | 1M nodes |
| **Adjacency list** | `list[list[(int, float)]]` | 1M nodes |
| **Dense** | `numpy.ndarray` | 5K nodes (dense 5K×5K ≈ 200MB — 1M×1M would be 8TB) |

In [1]:
import time
import tracemalloc
import numpy as np
import networkx as nx
import scipy.sparse as sp
import fastrp
import logging

logging.basicConfig(level=logging.DEBUG)

In [2]:
def benchmark(label, fn):
    """Run fn(), report wall-clock time and peak Python-side memory."""
    tracemalloc.start()
    t0 = time.perf_counter()
    result = fn()
    t1 = time.perf_counter()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f"[{label}]")
    print(f"  Shape:       {result.shape}")
    print(f"  Time:        {t1 - t0:.3f}s")
    print(f"  Peak memory: {peak / 1e6:.1f} MB")
    print()
    return result, t1 - t0, peak

## 1. Generate the large graph (1M nodes)

In [3]:
N_LARGE = 1_000_000
EDGE_PROB = 0.00001

print(f"Generating random graph with {N_LARGE:,} nodes (p={EDGE_PROB})...")
G_large = nx.fast_gnp_random_graph(N_LARGE, EDGE_PROB, seed=42)
print(f"  Nodes: {G_large.number_of_nodes():,}  Edges: {G_large.number_of_edges():,}")

Generating random graph with 1,000,000 nodes (p=1e-05)...
  Nodes: 1,000,000  Edges: 5,001,371


### 1a.  Prepare CSR input

In [4]:
print("Converting to scipy CSR...")
adj_csr = nx.to_scipy_sparse_array(G_large, format="csr")
print(f"  CSR nnz: {adj_csr.nnz:,}")

Converting to scipy CSR...
  CSR nnz: 10,002,742


### 1b.  Prepare adjacency list input

In [5]:
print("Building adjacency list...")
t0 = time.perf_counter()
adj_list = [[(int(v), 1.0) for v in G_large.neighbors(u)] for u in range(N_LARGE)]
t1 = time.perf_counter()
total_edges = sum(len(nbrs) for nbrs in adj_list)
print(f"  Edges (directed): {total_edges:,}  Build time: {t1 - t0:.2f}s")

# Free the networkx graph — we don't need it anymore
del G_large

Building adjacency list...
  Edges (directed): 10,002,742  Build time: 5.81s


## 2. Benchmark CSR path (1M nodes)

In [6]:
emb_csr_u, t_csr_u, mem_csr_u = benchmark(
    "fit_csr (1M nodes)",
    lambda: fastrp.fit_csr(
        adj_csr.indptr, adj_csr.indices, adj_csr.data,
        adj_csr.shape[0], dim=128, weights=[0.0, 1.0, 1.0], seed=42, undirected=True
    )
)

[fit_csr (1M nodes)]
  Shape:       (1000000, 128)
  Time:        2.588s
  Peak memory: 80.7 MB



In [7]:
emb_csr, t_csr, mem_csr = benchmark(
    "fit_csr (1M nodes)",
    lambda: fastrp.fit_csr(
        adj_csr.indptr, adj_csr.indices, adj_csr.data,
        adj_csr.shape[0], dim=128, weights=[0.0, 1.0, 1.0], seed=42, undirected=False
    )
)

[fit_csr (1M nodes)]
  Shape:       (1000000, 128)
  Time:        1.677s
  Peak memory: 80.7 MB



## 3. Benchmark adjacency list path (1M nodes)

In [8]:
emb_adj_u, t_adj_u, mem_adj_u = benchmark(
    "fit_adj_list (1M nodes)",
    lambda: fastrp.fit_adj_list(adj_list, dim=128, weights=[0.0, 1.0, 1.0], seed=42, undirected=True)
)

[fit_adj_list (1M nodes)]
  Shape:       (1000000, 128)
  Time:        4.274s
  Peak memory: 0.7 MB



In [9]:
emb_adj, t_adj, mem_adj = benchmark(
    "fit_adj_list (1M nodes)",
    lambda: fastrp.fit_adj_list(adj_list, dim=128, weights=[0.0, 1.0, 1.0], seed=42, undirected=False)
)

[fit_adj_list (1M nodes)]
  Shape:       (1000000, 128)
  Time:        3.303s
  Peak memory: 0.7 MB



## 4. Benchmark dense path (5K nodes)

A 1M×1M dense `float64` matrix would be ~8TB. We use a 5K node graph instead.

In [10]:
N_SMALL = 5_000

print(f"Generating dense graph with {N_SMALL:,} nodes...")
G_small = nx.fast_gnp_random_graph(N_SMALL, 0.005, seed=42)
print(f"  Nodes: {G_small.number_of_nodes():,}  Edges: {G_small.number_of_edges():,}")

adj_dense = nx.to_numpy_array(G_small, dtype=np.float64)
print(f"  Dense matrix: {adj_dense.shape}, {adj_dense.nbytes / 1e6:.1f} MB")
del G_small

Generating dense graph with 5,000 nodes...
  Nodes: 5,000  Edges: 62,381
  Dense matrix: (5000, 5000), 200.0 MB


In [11]:
emb_dense, t_dense, mem_dense = benchmark(
    "fit_dense (5K nodes)",
    lambda: fastrp.fit_dense(adj_dense, dim=128, weights=[0.0, 1.0, 1.0], seed=42, undirected=True)
)

[fit_dense (5K nodes)]
  Shape:       (5000, 128)
  Time:        0.105s
  Peak memory: 0.7 MB



## 5. Benchmark FastRP sklearn estimator (CSR via auto-detect)

Verifies that the estimator wrapper correctly dispatches to the CSR path.

In [12]:
emb_est, t_est, mem_est = benchmark(
    "FastRP.fit_transform (1M nodes, CSR auto-detect)",
    lambda: fastrp.FastRP(dim=128, weights=[0.0, 1.0, 1.0], seed=42, undirected=True).fit_transform(adj_csr)
)

DEBUG:fastrp:Fitting FastRP from CSR matrix...


[FastRP.fit_transform (1M nodes, CSR auto-detect)]
  Shape:       (1000000, 128)
  Time:        2.542s
  Peak memory: 80.3 MB



## 6. Summary

In [13]:
print(f"{'Path':<45} {'Nodes':>10} {'Time (s)':>10} {'Peak Mem (MB)':>14}")
print("-" * 82)
for label, nodes, t, mem in [
    ("fit_csr", f"{N_LARGE:,}", t_csr, mem_csr),
    ("fit_csr_undirect", f"{N_LARGE:,}", t_csr_u, mem_csr_u),
    ("fit_adj_list", f"{N_LARGE:,}", t_adj, mem_adj),
    ("fit_adj_list_undirect", f"{N_LARGE:,}", t_adj_u, mem_adj_u),
    ("fit_dense", f"{N_SMALL:,}", t_dense, mem_dense),
    ("FastRP.fit_transform (CSR auto)", f"{N_LARGE:,}", t_est, mem_est),
]:
    print(f"{label:<45} {nodes:>10} {t:>10.3f} {mem/1e6:>14.1f}")

Path                                               Nodes   Time (s)  Peak Mem (MB)
----------------------------------------------------------------------------------
fit_csr                                        1,000,000      1.677           80.7
fit_csr_undirect                               1,000,000      2.588           80.7
fit_adj_list                                   1,000,000      3.303            0.7
fit_adj_list_undirect                          1,000,000      4.274            0.7
fit_dense                                          5,000      0.105            0.7
FastRP.fit_transform (CSR auto)                1,000,000      2.542           80.3
